**WEEK 5 SIMULATION NOTEBOOK**

Continuing on from Week 4, this week, the objectives are: 

1. Simulating the behaviour of the $\Pi$ matrices as $\varepsilon$ tends to zero, and how the error changes. 
2. Simulating the output of non-decaying signals. 
3. Simulaitng the output when $S_{11} = S_{22} = 0$

Some theoretical objectives are: 

1. Finding conditions for $\Pi_{z1}$, $\Pi_{z2}$, $\Pi_{x2}$,
2. What happens when $S_{22}$ in NOT Hurwitz?
3. Controllability and Observability, explore the connections? 

In terms of simulation, the main thing that needs to change is to be able to implement varying values of $\varepsilon$


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import scipy.linalg as la


In [ ]:
# array of small values between 0 and 1 for epsilon
epsilon_list = np.array( [10 ** (-n) for n in range(0, 6)])


array([1.e+00, 1.e-01, 1.e-02, 1.e-03, 1.e-04, 1.e-05])

Let us make it more general than before: 
\begin{align}
\dot{x} &= A_{11} x + A_{12} z + B_1 u \in \mathbb{R}^n \\
\varepsilon \dot{z} &= A_{21} x + A_{22} z + B_2 u \in \mathbb{R}^m
\end{align}

\begin{align}
\dot{w}_1 &= S_{11} w_1 + S_{12} w_2 \in \mathbb{R}^{N_{g1}} \\
\varepsilon \dot{w}_2 &= S_{21} w_1 + S_{22} w_1 \in \mathbb{R}^{N_{g2}}
\end{align}

\begin{equation}
u = L_{g1} w_1 + L_{g2} w_2 \in \mathbb{R}
\end{equation}

\begin{equation}
y = C_1 x + C_2 z \in \mathbb{R}
\end{equation}

Below is a general framework for solving the system dynamics for whatever system/ signal we might have. 


In [ ]:
from dataclasses import dataclass

@dataclass
class SystemParams:
    n: int
    m: int
    Ng1: int
    Ng2: int
    A11: np.ndarray
    A12: np.ndarray
    A21: np.ndarray
    A22: np.ndarray
    S11: np.ndarray
    S12: np.ndarray
    S21: np.ndarray
    S22: np.ndarray
    Lg1: np.ndarray
    Lg2: np.ndarray
    B1: np.ndarray
    B2: np.ndarray
    epsilon: float

def combined_system(t, state, p: SystemParams):
    # Unpack states
    x = state[:p.n]
    z = state[p.n:p.n+p.m]
    w1 = state[p.n+p.m:p.n+p.m+p.Ng1]
    w2 = state[p.n+p.m+p.Ng1:p.n+p.m+p.Ng1+p.Ng2]

    # Input
    u = p.Lg1 @ w1 + p.Lg2 @ w2

    # Dynamics
    dx = p.A11 @ x + p.A12 @ z + p.B1.flatten() * u
    dz = (p.A21 @ x + p.A22 @ z + p.B2.flatten() * u) / p.epsilon
    dw1 = p.S11 @ w1 + p.S12 @ w2
    dw2 = (p.S21 @ w1 + p.S22 @ w2) / p.epsilon

    return np.concatenate([dx, dz, dw1, dw2])